# Schumann: review measure regions and link the facsimile

Session 2, **65–85 minutes**, followed by the **85–90 minute checkpoint**.
Run `02_pdf_preparation.ipynb` first and keep the same `schumann_edition_work`
project. This notebook uses `schumann-measure-boxes.zip`, a separate 2 KB file
containing **manual** teaching rectangles. No image is uploaded to a detector.

The worked example starts with one deliberately missing region: the pickup.
The supplied solution restores it, making 21 regions across five systems.
For the hands-on task, switch off `USE_PREPARED_SOLUTION`, enter your own
coordinates from the ruler, and check the overlay before recording approval.

Saved reference copies clear source-scan images and the image-bearing viewer;
run the notebook locally to inspect them. The musical starter is preserved.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" verovio==6.3.0

In [1]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Manual region exercise and protected Schumann facsimile integration."""

from pathlib import Path
import hashlib
import json
import os
import tempfile
import xml.etree.ElementTree as ET
import zipfile


def _link_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _link_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def _link_write_once(path, data):
    path = Path(path)
    if path.exists():
        if path.read_bytes() != data:
            raise FileExistsError(f"Preserving different file: {path}. Use a new attempt folder.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("xb") as output:
            output.write(data)


def find_schumann_box_bundle(start=None):
    start = Path(start or Path.cwd()).resolve()
    name = "schumann-measure-boxes.zip"
    for folder in (start, *start.parents):
        for candidate in (folder / name, folder / "notebooks" / name,
                          folder / "sessions/02-building-an-edition/notebooks" / name):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Copy {name} beside the notebook or run get_notebooks.py.")


def load_schumann_box_context(project, bundle):
    project = Path(project).resolve()
    checkpoint_file = project / "page-preparation-checkpoint.json"
    if not checkpoint_file.exists():
        raise FileNotFoundError("Run 02_pdf_preparation.ipynb first, or set PROJECT to that notebook's saved project.")
    checkpoint = json.loads(checkpoint_file.read_text())
    mei, pages = project / checkpoint["mei_input"], project / checkpoint["pages_json"]
    if _link_hash(mei) != checkpoint["mei_sha256"] or _link_hash(pages) != checkpoint["pages_json_sha256"]:
        raise ValueError("The page-preparation inputs changed; review that checkpoint before linking.")
    page = json.loads(pages.read_text())["pages"][0]
    with zipfile.ZipFile(bundle) as archive:
        expected = json.loads(archive.read("manifest.json"))["files_sha256"]
        if set(expected) != {"boxes-checked.json", "boxes-exercise.json"}:
            raise ValueError("Unexpected manual box bundle.")
        values = {}
        for name, digest in expected.items():
            data = archive.read(name)
            if hashlib.sha256(data).hexdigest() != digest:
                raise ValueError(f"Box data hash mismatch: {name}")
            values[name] = json.loads(data)
    checked, exercise = values["boxes-checked.json"], values["boxes-exercise.json"]
    image = pages.parent / page["image"]
    for data in (checked, exercise):
        if data["image_sha256"] != page["sha256"] or _link_hash(image) != data["image_sha256"]:
            raise ValueError("Manual regions belong to different source pixels; review the rendering before reusing them.")
        if data["image_pixels"] != [page["width"], page["height"]]:
            raise ValueError("Manual regions have different image geometry.")
    return {"project": project, "mei": mei, "pages_json": pages, "page": page,
            "image": image, "checked": checked, "exercise": exercise,
            "alignment": project / "edition/alignment.json"}


def _native_boxes(data):
    return {"measures": [{key: box[key] for key in ("ulx", "uly", "lrx", "lry")}
                         for box in data["measures"]]}


def diagnose_missing_pickup(context):
    """Exercise the real alignment gate in isolation; keep working boxes intact."""
    from camat.pdf_facsimile import build_measure_alignment
    manifest = json.loads(context["pages_json"].read_text())
    with tempfile.TemporaryDirectory(prefix="schumann-box-exercise-") as folder:
        folder = Path(folder)
        manifest["pages"][0]["image"] = os.path.relpath(context["image"], folder)
        _link_json(folder / "pages.json", manifest)
        _link_json(folder / "boxes" / f"{context['page']['page_id']}.json", _native_boxes(context["exercise"]))
        try:
            build_measure_alignment(folder / "pages.json", output=folder / "alignment.json")
        except ValueError as problem:
            message = str(problem)
            if "actual [4, 4, 4, 4, 4], expected [5, 4, 4, 4, 4]" not in message:
                raise
        else:
            raise AssertionError("The deliberately incomplete boxes unexpectedly passed.")
    _link_json(context["project"] / "edition/exercise-diagnostic.json", {
        "source": "Manually prepared teaching data; no detector called.", "status": "rejected as expected", "message": message})
    return message


def show_pickup_ruler(context):
    from PIL import Image
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import MultipleLocator
    crop = (320, 560, 1750, 1400)
    with Image.open(context["image"]) as image:
        region = image.crop(crop).convert("RGB")
    figure, axes = plt.subplots(figsize=(13, 7))
    axes.imshow(region, extent=(crop[0], crop[2], crop[3], crop[1]))
    for index, box in enumerate(context["exercise"]["measures"]):
        if box["system"] != 1:
            continue
        axes.add_patch(Rectangle((box["ulx"], box["uly"]), box["lrx"] - box["ulx"], box["lry"] - box["uly"],
                                 fill=False, edgecolor="tab:red", linewidth=2))
        axes.text(box["ulx"] + 12, box["uly"] + 12, f"box {index} / MEI {box['mei_label']}", va="top",
                  color="tab:red", bbox={"facecolor": "white", "edgecolor": "none"})
    axes.set(xlim=(crop[0], crop[2]), ylim=(crop[3], crop[1]),
             xlabel="Full-image x coordinate (pixels)", ylabel="Full-image y coordinate (pixels)",
             title="Missing pickup region: both staves, before box 0")
    axes.xaxis.set_major_locator(MultipleLocator(100)); axes.yaxis.set_major_locator(MultipleLocator(100))
    axes.grid(alpha=0.25)
    figure.tight_layout()
    output = context["project"] / "edition/previews/pickup-ruler.png"
    output.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output, dpi=120)
    plt.show()
    plt.close(figure)


def correct_schumann_boxes(context, additions):
    from camat.pdf_facsimile import edit_measure_boxes, build_measure_alignment, draw_alignment_overlays
    path = context["pages_json"].parent / "boxes" / f"{context['page']['page_id']}.json"
    backup = path.with_name(path.stem + ".original.json")
    baseline = (json.dumps(_native_boxes(context["exercise"]), ensure_ascii=False, indent=2) + "\n").encode()
    if backup.exists():
        if backup.read_bytes() != baseline:
            raise ValueError("Working box baseline differs from this exercise; preserve it and use a new attempt.")
    else:
        _link_write_once(path, baseline)
    edit_measure_boxes(context["pages_json"], context["page"]["page_id"], add=additions)
    build_measure_alignment(context["pages_json"], output=context["alignment"])
    return draw_alignment_overlays(context["alignment"], context["project"] / "edition/overlays")


def _score_projection(path):
    ns = "{http://www.music-encoding.org/ns/mei}"
    body = ET.parse(path).getroot().find(ns + "music/" + ns + "body")
    def walk(element):
        if element.tag in {ns + "pb", ns + "sb"}:
            return None
        attributes = {k: v for k, v in element.attrib.items() if k != "facs"}
        if element.tag == ns + "mdiv":
            attributes.pop("n", None)  # The single unnumbered movement is explicitly numbered 1.
        return (element.tag, sorted(attributes.items()), (element.text or "").strip(),
                [value for child in element if (value := walk(child)) is not None])
    return walk(body)


def create_linked_schumann(context):
    """Generate in the final folder so relative graphic paths survive promotion."""
    from camat.pdf_facsimile import number_single_mdiv, integrate_pdf_facsimile, write_page_breaks
    folder = context["project"] / "edition/mei"
    folder.mkdir(parents=True, exist_ok=True)
    handles = [tempfile.NamedTemporaryFile(prefix=".schumann-", suffix=".mei", dir=folder, delete=False) for _ in range(2)]
    staged_input, candidate = [Path(handle.name) for handle in handles]
    for handle in handles:
        handle.close()
    baseline = folder / "linked-starter.mei"
    try:
        staged_input.write_text(number_single_mdiv(context["mei"].read_text()), encoding="utf-8")
        report = integrate_pdf_facsimile(staged_input, context["pages_json"], context["alignment"], candidate,
                                         source_ref="src.imslp51556")
        breaks = write_page_breaks(candidate, context["pages_json"], candidate)
        if _score_projection(context["mei"]) != _score_projection(candidate):
            raise ValueError("Linking changed musical content or existing IDs.")
        _link_write_once(baseline, candidate.read_bytes())
    finally:
        staged_input.unlink(missing_ok=True)
        candidate.unlink(missing_ok=True)
    report["output_mei"] = str(baseline)
    breaks["output_mei"] = str(baseline)
    _link_json(context["project"] / "edition/linking-report.json", {"integration": report, "breaks": breaks,
               "score_projection_unchanged": True, "controlled_change": "Number the sole unnumbered mdiv 1; add facsimile links and source breaks."})
    snapshot = context["project"] / "edition/checkpoints/linked-before-editing.mei"
    _link_write_once(snapshot, baseline.read_bytes())
    work = folder / "work.mei"
    created = not work.exists()
    if created:
        _link_write_once(work, baseline.read_bytes())
    checkpoint = {"stage": "linked teaching draft before expressive editing", "project": str(context["project"]),
                  "input_mei_sha256": _link_hash(context["mei"]), "linked_mei": "edition/mei/linked-starter.mei",
                  "linked_mei_sha256": _link_hash(baseline), "work_mei": "edition/mei/work.mei",
                  "work_created_now": created, "work_matches_linked_baseline": work.read_bytes() == baseline.read_bytes(),
                  "work_sha256_at_check": _link_hash(work), "snapshot": "edition/checkpoints/linked-before-editing.mei",
                  "snapshot_sha256": _link_hash(snapshot), "image_sha256": context["page"]["sha256"],
                  "alignment_sha256": _link_hash(context["alignment"]), "unreviewed_pages": report["unreviewed_pages"],
                  "measures_linked": report["measures_linked"], "zones": report["zones"], "surfaces": report["surfaces"],
                  "pb": breaks["pb"], "sb": breaks["sb"], "musical_content_and_existing_ids_preserved": True,
                  "scope": "Labels 1–9 are the core editorial passage; full-piece musical proofreading remains incomplete."}
    _link_json(context["project"] / "linked-stage-checkpoint.json", checkpoint)
    return baseline, work, checkpoint

In [2]:
import subprocess
import sys
from IPython.display import display, Image
from camat.pdf_facsimile import approve_alignment_review

PROJECT = Path.cwd() / "schumann_edition_work"  # Same folder as in PDF preparation.
CONTEXT = load_schumann_box_context(PROJECT, find_schumann_box_bundle())
MEI_HASH_BEFORE = _link_hash(CONTEXT["mei"])

USE_PREPARED_SOLUTION = True  # False = try your coordinates in the next settings cell.
REVIEWER = ""               # Your name after inspecting all regions.
CONFIRM_BOXES = False       # True only after that inspection; no approval is assumed.
OPEN_VIEWER = True

print("Input:", CONTEXT["mei"])
print("Page manifest:", CONTEXT["pages_json"])
print("Manual regions; source image:", CONTEXT["page"]["sha256"])

Input: /tmp/camat-schumann-linking/notebook/schumann_edition_work/inputs/b02b5197b383/starter.mei
Page manifest: /tmp/camat-schumann-linking/notebook/schumann_edition_work/edition/pages/pages.json
Manual regions; source image: bfeb442d31808cb610fc85ee1e0d2e460d70049141abcfae7660f917ab5bdaec


## 65–70 — Find the missing pickup

The manual reference has 21 regions; the exercise version intentionally
omits MEI label **1**, the short upbeat before the first full bar. The first
system therefore has four boxes instead of five. This simulates a correction
task; it is not a measured detector failure.

Run the real CAMAT alignment check on an isolated exercise copy. It rejects
the mismatch without changing your working boxes or an earlier alignment.

In [3]:
print(diagnose_missing_pickup(CONTEXT))

System count gate failed; review boxes and pages.json:
schumann-op68-no10-source-page-p0001 (PDF page 1, printed page 8): actual [4, 4, 4, 4, 4], expected [5, 4, 4, 4, 4]


In [4]:
show_pickup_ruler(CONTEXT)

## 70–75 — Restore the region, then inspect the whole page

Rectangles are `(left, top, right, bottom)` in **full PNG pixels**, with the
origin at the top-left. Cover both piano staves and the pickup note/rest.
Copy top/bottom from the neighbouring region and read left/right from the
ruler. The on-screen preview is scaled; its display width is not the coordinate
system. Boxes may overlap slightly at barlines.

For the exercise, set `USE_PREPARED_SOLUTION=False` above and fill the list
below. The prepared solution is the default for a complete Run All/reference
demonstration; its use is reported explicitly. An incorrect/missing region
prevents alignment until corrected.

In [5]:
TRY_YOUR_BOXES = [
    # (left, top, right, bottom),
]

if USE_PREPARED_SOLUTION:
    pickup = CONTEXT["checked"]["measures"][0]
    ADD_BOXES = [tuple(pickup[key] for key in ("ulx", "uly", "lrx", "lry"))]
    print("Using the prepared pickup solution:", ADD_BOXES)
else:
    ADD_BOXES = TRY_YOUR_BOXES

OVERLAYS = correct_schumann_boxes(CONTEXT, ADD_BOXES)
alignment = json.loads(CONTEXT["alignment"].read_text())
print("Aligned regions:", len(alignment["regions"]))
print("Regions per system:", [sum(r["system"] == number for r in alignment["regions"]) for number in range(1, 6)])

Using the prepared pickup solution: [(400, 600, 872, 1360)]


Aligned regions: 21
Regions per system: [5, 4, 4, 4, 4]


In [6]:
for path in OVERLAYS:
    display(Image(filename=str(path), width=1100))

Check **all regions**, especially the pickup and the first/last bar of each
system. The labels are `movement/MEI label`; `1/2` is the first full bar here.
A rectangle is a source region, not a segmentation of every individual symbol:
slurs and text may cross barlines. Consult the full page when a crop truncates
a cross-bar marking.

A matching count cannot prove that the regions cover the correct music.
The defaults leave the review **unreviewed**. After inspection, enter your
name, set `CONFIRM_BOXES=True`, rerun the settings cell and continue below.
This approval concerns geometry/alignment, not a complete musical proofread.

In [7]:
if CONFIRM_BOXES:
    if not REVIEWER.strip():
        raise ValueError("Enter the name of the person who inspected the overlay.")
    approve_alignment_review(CONTEXT["alignment"], REVIEWER)
    print("Alignment review recorded for:", REVIEWER)
else:
    print("Alignment remains unreviewed. A linked draft can be inspected; no human approval is claimed.")

Alignment remains unreviewed. A linked draft can be inspected; no human approval is claimed.


## 75–80 — Combine the musical MEI and the regions

The merge adds a **facsimile** with one **surface**, one page **graphic** and
21 measure **zones**. Each musical `measure/@facs` points to its zone. A
page break points to the surface; four system breaks follow the print.
The sole unnumbered movement is explicitly numbered 1 for alignment.

The generated baseline is `edition/mei/linked-starter.mei`. A separate
`edition/mei/work.mei` becomes the editorial working file. Both find the
page image by `../pages/page-0001.png`. Generation occurs in that same folder,
so promoting a copy does not break relative paths. Existing working edits are
preserved; regenerate in a new attempt folder if the baseline itself changes.

In [8]:
LINKED_MEI, WORK_MEI, CHECKPOINT = create_linked_schumann(CONTEXT)
assert _link_hash(CONTEXT["mei"]) == MEI_HASH_BEFORE
assert (CHECKPOINT["measures_linked"], CHECKPOINT["zones"], CHECKPOINT["surfaces"]) == (21, 21, 1)
assert (CHECKPOINT["pb"], CHECKPOINT["sb"]) == (1, 4)
print("Linked baseline:", LINKED_MEI)
print("Editorial working file:", WORK_MEI)
print("Input musical content and existing IDs preserved.")
print("Unreviewed pages:", CHECKPOINT["unreviewed_pages"])

Linked baseline: /tmp/camat-schumann-linking/notebook/schumann_edition_work/edition/mei/linked-starter.mei
Editorial working file: /tmp/camat-schumann-linking/notebook/schumann_edition_work/edition/mei/work.mei
Input musical content and existing IDs preserved.
Unreviewed pages: ['schumann-op68-no10-source-page-p0001']


In [9]:
ns = {"m": "http://www.music-encoding.org/ns/mei"}
root = ET.parse(LINKED_MEI).getroot()
surface = root.find("m:music/m:facsimile/m:surface", ns)
first_measure = root.find("m:music/m:body//m:measure", ns)
page_break = root.find("m:music/m:body//m:pb", ns)
for label, element in [("surface", surface), ("graphic", surface.find("m:graphic", ns)),
                       ("first zone", surface.find("m:zone", ns)),
                       ("first musical measure", first_measure), ("page break", page_break)]:
    print(label, dict(element.attrib))

surface {'{http://www.w3.org/XML/1998/namespace}id': 'surface-schumann-op68-no10-source-page-p0001', 'n': '8', 'ulx': '0', 'uly': '0', 'lrx': '4166', 'lry': '5616'}
graphic {'{http://www.w3.org/XML/1998/namespace}id': 'graphic-schumann-op68-no10-source-page-p0001', 'target': '../pages/page-0001.png', 'width': '4167', 'height': '5617'}
first zone {'{http://www.w3.org/XML/1998/namespace}id': 'zone-schumann-op68-no10-source-page-p0001-001', 'type': 'measure', 'ulx': '400', 'uly': '600', 'lrx': '872', 'lry': '1360', 'label': '1/1'}
first musical measure {'n': '1', '{http://www.w3.org/XML/1998/namespace}id': 'd1e9', 'type': 'upbeat', 'metcon': 'false', 'facs': '#zone-schumann-op68-no10-source-page-p0001-001'}
page break {'{http://www.w3.org/XML/1998/namespace}id': 'pb-schumann-op68-no10-source-page-p0001', 'n': '8', 'facs': '#surface-schumann-op68-no10-source-page-p0001'}


### Validate the linked baseline

The validator checks local references/resources, MEI 5.1 schema constraints,
Schematron and Verovio rendering. Read the reported checks separately:
the optional CAMAT parse consumer is not requested. Neither these checks nor
rendering establish source fidelity. The inherited rhythm diagnostics remain
visible rather than being silenced by changing notes or rests.

In [10]:
VALIDATION_FOLDER = PROJECT / "edition/validation-linked"
validation = subprocess.run([sys.executable, "-m", "camat.mei_validation", str(LINKED_MEI),
                             "--output-dir", str(VALIDATION_FOLDER), "--verovio"],
                            text=True, capture_output=True, timeout=180)
print(validation.stdout)
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError("Linked MEI validation failed; inspect the saved reports.")
validation_run = json.loads((VALIDATION_FOLDER / "run.json").read_text())
print("Conformance:", validation_run["conformance"])
print("Consumer diagnostics:", validation_run["consumer_diagnostics"])

MEI conformance: passed; consumers: incomplete

Conformance: passed
Consumer diagnostics: incomplete


## 80–85 — Check the score/source interaction

Hover and click a bar in either panel. Its partner should highlight; the
status line identifies its MEI measure and source zone. Check the pickup, a
system transition (labels 5/6) and the final bar. Open the saved local HTML
separately if your notebook frontend requires trust before running scripts.
It contains the local scan for preview and stays inside the working project.

In [11]:
if OPEN_VIEWER:
    from camat.facsimile_viewer import build_facsimile_viewer, embed_viewer_html
    PREVIEW = build_facsimile_viewer(
        WORK_MEI, verovio_options={"breaks": "encoded", "header": "none", "footer": "none"},
        allow_missing_facsimile=False, show_diagnostic_table=True, show_annotations=False,
    )
    local_html = PROJECT / "edition/previews/facsimile-viewer.html"
    local_html.parent.mkdir(parents=True, exist_ok=True)
    local_html.write_text('<!doctype html><html lang="en"><head><meta charset="utf-8">'
                          '<title>Schumann facsimile exercise</title></head><body>'
                          + PREVIEW["html"] + '</body></html>', encoding="utf-8")
    print(PREVIEW["summary"])
    print("Local preview:", local_html)
    display({"text/html": embed_viewer_html(PREVIEW["html"], min_height=900)}, raw=True)

MEI:             schumann_edition_work/edition/mei/work.mei
Viewer mode:     score + facsimile
Surfaces:        1
Measures:        21
Linked zones:    21
Missing @facs:   0
Annotations:     1
Layout hints:    0 missing <pb>, 0 missing <sb>
Local preview: /tmp/camat-schumann-linking/notebook/schumann_edition_work/edition/previews/facsimile-viewer.html


## 85–90 — Save the unedited linked stage

The merge also saves `edition/checkpoints/linked-before-editing.mei` and
`linked-stage-checkpoint.json`. Keep them with the page images, alignment,
review and validation reports. If `work.mei` already contains edits, rerunning
this notebook preserves them; the saved baseline remains the comparison point.

Write one sentence about the automated checks and one about your visual
review. Then continue editing **work.mei** in the next block; do not regenerate
the musical file from the facsimile workflow after adding editorial content.

In [12]:
print("Saved snapshot:", PROJECT / CHECKPOINT["snapshot"])
print("Snapshot SHA-256:", CHECKPOINT["snapshot_sha256"])
print("Checkpoint:", PROJECT / "linked-stage-checkpoint.json")
print("Keep the whole edition folder together, including pages/ and mei/.")
print("Core editorial passage: MEI labels 1–9; the rest remains a technical carrier.")

Saved snapshot: /tmp/camat-schumann-linking/notebook/schumann_edition_work/edition/checkpoints/linked-before-editing.mei
Snapshot SHA-256: ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885
Checkpoint: /tmp/camat-schumann-linking/notebook/schumann_edition_work/linked-stage-checkpoint.json
Keep the whole edition folder together, including pages/ and mei/.
Core editorial passage: MEI labels 1–9; the rest remains a technical carrier.


This exercise uses CAMAT's public PDF integration functions and manually
prepared regions. A live detector is an optional alternative after explaining
the image upload and preserving its actual response separately. The detector
proposes regions, not musical notes. The saved reference notebook contains
code and text checks, without source-image or viewer payloads.